WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
from typing import Tuple

import chex
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import trimesh

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)

### Design variable packing

In [ ]:
@chex.dataclass
class DesignExtents:
  """Physical ranges for every magnetic design variable."""

  remnant_polar: _utils.Extent
  remnant_azith: _utils.Extent
  remnant_strength: _utils.Extent
  applied_polar: _utils.Extent
  applied_azith: _utils.Extent
  applied_strength: _utils.Extent

In [ ]:
def design_vars_to_opt_vars(
  design_vars: jnp.ndarray,
  extents: DesignExtents,
  num_poses: int,
  num_elems: int,
  num_fibers: int,
) -> Tuple[jnp.ndarray, jnp.ndarray]:
  """Expand the normalized MMA design vector into physical magnetic flux fields.

  The design vector is a single flat array in `[0, 1]`

      [ remanent block : 3 * num_elems * num_fibers ][ applied block : 3 * num_poses ]

  The remanent block carries `(polar, azimuthal, strength)` for each fiber of each
  element. The applied block reshapes to `(num_poses, 3)`.

  Args:
    design_vars: Array of shape `(num_design_var,)` with entries in `[0, 1]`.
    extents: `DesignExtents` supplying the physical range of each field.
    num_poses: Number of distinct applied field configurations.
    num_elems: Number of rod elements.
    num_fibers: Number of fibers per element cross-section.

  Returns:
    A tuple `(remnant_mag_flux, applied_mag_flux)` of shapes
      `(num_elems, num_fibers, num_dim=3)` [T] and `(num_poses, num_dim=3)` [T].
  """
  num_remnant_vars = 3 * num_elems * num_fibers
  remnant_block = design_vars[:num_remnant_vars].reshape(num_elems, num_fibers, 3)
  applied_block = design_vars[num_remnant_vars:].reshape(num_poses, 3)

  remnant_mag_flux = _utils.polar_azith_to_vector(
    extents.remnant_polar.renormalize_array(remnant_block[..., 0]),
    extents.remnant_azith.renormalize_array(remnant_block[..., 1]),
    extents.remnant_strength.renormalize_array(remnant_block[..., 2]),
  )
  applied_mag_flux = _utils.polar_azith_to_vector(
    extents.applied_polar.renormalize_array(applied_block[:, 0]),
    extents.applied_azith.renormalize_array(applied_block[:, 1]),
    extents.applied_strength.renormalize_array(applied_block[:, 2]),
  )
  return remnant_mag_flux, applied_mag_flux

In [ ]:
def opt_vars_to_design_vars(
  remnant_mag_flux: jnp.ndarray,
  applied_mag_flux: jnp.ndarray,
  extents: DesignExtents,
) -> jnp.ndarray:
  """Collapse physical magnetic flux fields into the normalized MMA design vector.

  Inverse of `design_vars_to_opt_vars`. The round trip only closes when every
  physical value already lies inside its extent; values outside normalize beyond
  `[0, 1]` and are left unclipped so an out-of-range guess stays visible.

  Args:
    remnant_mag_flux: Array of shape `(num_elems, num_fibers, num_dim=3)` [T].
    applied_mag_flux: Array of shape `(num_poses, num_dim=3)` [T].
    extents: `DesignExtents` supplying the physical range of each field.

  Returns:
    Array of shape `(num_design_var,)` of normalized design variables.
  """
  remnant_polar, remnant_azith, remnant_strength = _utils.vector_to_polar_azith(
    remnant_mag_flux
  )
  applied_polar, applied_azith, applied_strength = _utils.vector_to_polar_azith(
    applied_mag_flux
  )
  remnant_block = jnp.stack(
    [
      extents.remnant_polar.normalize_array(remnant_polar),
      extents.remnant_azith.normalize_array(remnant_azith),
      extents.remnant_strength.normalize_array(remnant_strength),
    ],
    axis=-1,
  )
  applied_block = jnp.stack(
    [
      extents.applied_polar.normalize_array(applied_polar),
      extents.applied_azith.normalize_array(applied_azith),
      extents.applied_strength.normalize_array(applied_strength),
    ],
    axis=-1,
  )
  return jnp.concatenate([remnant_block.reshape(-1), applied_block.reshape(-1)])

### Geometry

In [ ]:
rod_length = 0.020
num_elems = 40
num_nodes = num_elems + 1
fiber_offset = 4.0e-4
fiber_radius = 2.0e-4
shear_modulus = 3.0e5

offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0], [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = 4
area_per_fiber = np.pi * fiber_radius**2

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda k: jnp.array([k * rod_length, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, 1.0, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)
contact_radii = np.asarray(rod.contact_radii)

In [ ]:
APPLIED_FLUX_MAX = 8.0e-3
TARGET_TIP_DEFLECTION = 14.0e-3

bending_stiffness = (
  3.0 * shear_modulus * np.sum(area_per_fiber * offsets_local[:, 0] ** 2)
)
remnant_scale = (
  TARGET_TIP_DEFLECTION
  * 3.0
  * bending_stiffness
  * _utils.PERMEABILITY_VACUUM
  / (rod_length**3 * num_fibers * area_per_fiber * APPLIED_FLUX_MAX)
)
print(f"B_r scale ~ {remnant_scale:.4f} T  (want ~0.1 T)")

### Obstacle

In [ ]:
# The box bottom face sits on the unobstructed path so contact fires immediately.
# The rod centreline is held one contact radius below it, forcing the optimizer to
# route around rather than through.
# A wide shelf rather than a narrow bump. A short obstacle only makes the rod dip
# and carry on, which a single uniform magnetization already achieves. Pinning a
# long stretch flat and leaving the tip free forces low curvature under the shelf
# and high curvature past it, which is what a spatially varying magnetization can
# supply and a uniform one cannot.
obstacle_span_x = 9.0e-3
obstacle_span_y = 4.0e-3
obstacle_centre = np.array([0.0095, 0.0030, 0.0])
box = trimesh.creation.box(extents=(obstacle_span_x, obstacle_span_y, 24.0e-3))
box.apply_translation(obstacle_centre)

wall_penalty_stiffness = 1.0e3 * bending_stiffness / (rod_length / num_elems) ** 4
obstacle = _contact.Obstacle.from_mesh(
  mesh=box,
  grid_shape=(48, 48, 32),
  wall_penalty_stiffness=wall_penalty_stiffness,
  padding=3.0 * contact_radii.max(),
  extrap_fill_value=10.0,
)

### Design extents and the three configurations

In [ ]:
LOCK = 1.0e-6
IN_PLANE_POLAR = 0.5 * np.pi
FIXED_APPLIED_AZITH = np.radians(45.0)
FIXED_REMNANT_AZITH = 0.0
mpi_ppi = _utils.Extent(min=-np.pi, max=np.pi)


def locked(value):
  """An Extent pinning a design variable to a fixed physical value."""
  return _utils.Extent(min=value - LOCK, max=value + LOCK)


def build_extents(free_remnant: bool, free_applied: bool) -> DesignExtents:
  """Extents for one configuration.

  Polar angles are pinned to pi/2 throughout, placing all magnetization in the x-y
  plane where the rod bends. Azimuth is then the single free angle and sweeps the
  full circle without approaching the spherical singularity at the poles.
  """
  return DesignExtents(
    remnant_polar=locked(IN_PLANE_POLAR),
    remnant_azith=(mpi_ppi if free_remnant else locked(FIXED_REMNANT_AZITH)),
    remnant_strength=(
      _utils.Extent(min=0.2 * remnant_scale, max=remnant_scale)
      if free_remnant
      else locked(remnant_scale)
    ),
    applied_polar=locked(IN_PLANE_POLAR),
    applied_azith=(mpi_ppi if free_applied else locked(FIXED_APPLIED_AZITH)),
    applied_strength=locked(APPLIED_FLUX_MAX),
  )


CONFIGS = {
  "stimulus_only": build_extents(free_remnant=False, free_applied=True),
  "materials_only": build_extents(free_remnant=True, free_applied=False),
  "materials_stimulus": build_extents(free_remnant=True, free_applied=True),
}

### Problem

In [ ]:
num_poses = 1
num_load_steps = 25
num_design_var = 3 * num_elems * num_fibers + 3 * num_poses
shear_moduli = jnp.full((num_elems, num_fibers), shear_modulus)

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
boundary_conditions = _bc.process_boundary_conditions(
  rod,
  [
    _bc.DirichletBC(
      values=(dof, [np.array([0.0]) for _ in range(num_load_steps)]),
      node_targets=[np.array([0]) for _ in range(num_load_steps)],
      name=f"clamp_{dof}",
    )
    for dof in clamped_dofs
  ]
  + [
    _bc.DirichletBC(
      values=(dof, [np.zeros(num_nodes) for _ in range(num_load_steps)]),
      node_targets=[np.arange(num_nodes) for _ in range(num_load_steps)],
      name=f"planar_{dof}",
    )
    for dof in (_utils.Translation.Z, _utils.Rotation.THETA_Y)
  ],
)
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 200,
    "threshold": 1e-6,
    "abs_threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 15,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-12},
)
problem = _fe.CosseratProblem(
  rod=rod, bc=boundary_conditions, nr_solver=nr_solver, obstacle=obstacle
)

In [ ]:
def solve_design(
  design_vars: jnp.ndarray, extents: DesignExtents
) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
  """Converged displacement vector and the magnetic fields that produced it.

  Args:
    design_vars: Array of shape `(num_design_var,)` with entries in `[0, 1]`.
    extents: `DesignExtents` for this configuration.

  Returns:
    A tuple `(u_star, remnant, applied)` of shapes `(num_dofs,)` [m, rad],
      `(num_elems, num_fibers, num_dim=3)` [T] and `(num_poses, num_dim=3)` [T].
  """
  remnant, applied = design_vars_to_opt_vars(
    design_vars=design_vars,
    extents=extents,
    num_poses=num_poses,
    num_elems=num_elems,
    num_fibers=num_fibers,
  )
  ramp = jnp.linspace(1.0 / num_load_steps, 1.0, num_load_steps)
  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=shear_moduli,
    remnant_mag_flux=remnant,
    applied_mag_flux=jnp.einsum("s, d -> sd", ramp, applied[0]),
  )
  return u_star, remnant, applied


def mean_tip_height(
  design_vars: jnp.ndarray, extents: DesignExtents
) -> Tuple[jnp.ndarray, Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]]:
  """Objective g = -mean(u_y)/L, with the converged state as an auxiliary output.

  Averaging over all nodes matches Wang's boundary average and is better
  conditioned than a single-node objective. Dividing by the rod length puts the
  objective near unity so MMA's move limits and KKT tolerance are meaningful.

  The converged state and both magnetic fields are returned alongside the objective
  so that anything needing the deformed shape or the magnetization pattern reuses
  this solve rather than repeating it.

  Returns:
    A tuple `(objective, (u_star, remnant, applied))`.
  """
  u_star, remnant, applied = solve_design(design_vars, extents)
  displacement = u_star.reshape(num_nodes, rod.num_dofs_per_node)[:, 1]
  return -jnp.mean(displacement) / rod_length, (u_star, remnant, applied)

### Live view

In [ ]:
def deformed_coords(u_star: jnp.ndarray) -> np.ndarray:
  """Deformed nodal coordinates [m] from a converged displacement vector."""
  nodal = np.asarray(u_star).reshape(num_nodes, rod.num_dofs_per_node)
  return np.asarray(rod.node_coords) + nodal[:, :3]


def draw_scene(
  ax,
  coords: np.ndarray,
  remnant: np.ndarray,
  applied: np.ndarray,
  title: str,
  arrow_stride: int = 4,
) -> None:
  """Draw a deformed shape with its magnetization pattern and applied field.

  The remanent arrows are anchored at element centres along the deformed rod and
  drawn in the lab frame, so they show the magnetization as it has been carried
  around by the deformation rather than as it was programmed. The applied field is
  drawn once as a single arrow in the upper left, at a length independent of its
  magnitude so that only its direction is read from the plot.

  Args:
    ax: Matplotlib axis, cleared before drawing.
    coords: Deformed node coordinates, shape `(num_nodes, num_dim=3)` [m].
    remnant: Remanent flux density, shape `(num_elems, num_fibers, num_dim=3)` [T].
    applied: Applied flux density, shape `(num_poses, num_dim=3)` [T].
    title: Axis title.
    arrow_stride: Draw one magnetization arrow every this many elements.
  """
  ax.clear()
  ax.add_patch(
    plt.Rectangle(
      (
        (obstacle_centre[0] - obstacle_span_x / 2) * 1e3,
        (obstacle_centre[1] - obstacle_span_y / 2) * 1e3,
      ),
      obstacle_span_x * 1e3,
      obstacle_span_y * 1e3,
      color="0.75",
      zorder=1,
    )
  )
  ax.plot(coords[:, 0] * 1e3, coords[:, 1] * 1e3, lw=2.5, color="0.2", zorder=2)

  centres = 0.5 * (coords[:-1] + coords[1:]) * 1e3
  fiber_mean = np.mean(remnant, axis=1)
  peak = max(float(np.linalg.norm(fiber_mean, axis=-1).max()), 1e-30)
  scale = rod_length * 1e3 * 0.09 / peak
  sel = slice(None, None, arrow_stride)
  ax.quiver(
    centres[sel, 0],
    centres[sel, 1],
    fiber_mean[sel, 0] * scale,
    fiber_mean[sel, 1] * scale,
    angles="xy",
    scale_units="xy",
    scale=1.0,
    width=0.005,
    color="crimson",
    zorder=3,
  )

  # Drawn in axes fractions rather than data coordinates so the arrow can never
  # be clipped by the data limits, whatever direction the optimizer picks.
  field_dir = applied[0, :2] / max(float(np.linalg.norm(applied[0, :2])), 1e-30)
  tail = np.array([0.18, 0.78])
  head = tail + 0.13 * field_dir
  ax.annotate(
    "",
    xy=tuple(head),
    xytext=tuple(tail),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "color": "royalblue", "lw": 2.0},
  )
  ax.text(
    0.02,
    0.90,
    f"$B_a$ {np.degrees(np.arctan2(field_dir[1], field_dir[0])):.0f} deg",
    transform=ax.transAxes,
    color="royalblue",
    fontsize=8,
  )
  ax.set_aspect("equal")
  ax.set_xlabel("$x$ (mm)")
  ax.set_ylabel("$y$ (mm)")
  ax.set_title(title, fontsize=9)

### Optimization

In [ ]:
def run_config(
  config_name: str,
  init_design_vars: jnp.ndarray = None,
  max_iter: int = 80,
  live_every: int = 0,
):
  """Optimize one configuration with MMA.

  Args:
    config_name: Key into `CONFIGS`.
    init_design_vars: Optional starting point of shape `(num_design_var,)`. Warm
      starting a richer configuration from the optimum of a poorer one guarantees
      the richer result is at least as good, since the poorer feasible set is a
      subset of the richer one.
    max_iter: Maximum MMA epochs.
    live_every: Redraw the shape every this many epochs. Zero disables the live
      view. The shape and both fields come from the auxiliary output of the
      objective, so a redraw costs no extra solve. It lags the design by one epoch.

  Returns:
    A tuple `(design_vars, history, state)` where `state` is
      `(u_star, remnant, applied)` from the last evaluated design, so callers need
      no further solve. It corresponds exactly to `history[-1]`.
  """
  extents = CONFIGS[config_name]
  # One compile per configuration. The extents are captured in the closure, so a
  # new configuration is a new function object and JAX compiles it again.
  objective_and_grad = jax.jit(
    jax.value_and_grad(lambda dv: mean_tip_height(dv, extents), has_aux=True)
  )
  start = (
    jnp.full((num_design_var,), 0.5) if init_design_vars is None else init_design_vars
  )

  mma_params = _mma.MMAParams(
    max_iter=max_iter,
    kkt_tol=1e-6,
    step_tol=1e-6,
    move_limit=0.05,
    num_design_var=num_design_var,
    num_cons=1,
    lower_bound=np.zeros((num_design_var, 1)),
    upper_bound=np.ones((num_design_var, 1)),
  )
  mma_state = _mma.init_mma(
    init_design_var=np.asarray(start).reshape(-1, 1), mma_params=mma_params
  )
  dummy_cons = np.full((1, 1), -1.0)
  dummy_grad_cons = np.zeros((1, num_design_var))
  history = []

  while not mma_state.is_converged and mma_state.epoch < max_iter:
    current = jnp.asarray(mma_state.x).reshape(-1)
    (obj_value, aux), obj_grad = objective_and_grad(current)
    u_star, remnant, applied = aux
    mma_state = _mma.update_mma(
      mma_state=mma_state,
      mma_params=mma_params,
      obj=np.array([float(obj_value)]),
      grad_obj=np.asarray(obj_grad, dtype=np.float64).reshape(-1, 1),
      cons=dummy_cons,
      grad_cons=dummy_grad_cons,
    )
    history.append(float(-obj_value) * rod_length)
    if live_every and mma_state.epoch % live_every == 0:
      # A fresh figure each time. The inline backend detaches a figure once it has
      # been shown, so redrawing into the same one updates nothing.
      fig, ax = plt.subplots(figsize=(6.5, 3.8))
      draw_scene(
        ax,
        deformed_coords(u_star),
        np.asarray(remnant),
        np.asarray(applied),
        f"{config_name}  epoch {mma_state.epoch}  "
        f"mean u_y = {history[-1] * 1e3:.3f} mm",
      )
      plt.show()
      plt.pause(1e-5)

  return jnp.asarray(mma_state.x).reshape(-1), history, (u_star, remnant, applied)

In [ ]:
# Stimulus only first, then warm start the two richer configurations from it. A
# configuration with more freedom cannot be worse than one with less at a global
# optimum, so a warm start converts that guarantee into a numerical one and
# removes the local minimum that otherwise makes co-design look inferior.
results = {}
design_s, history_s, state_s = run_config("stimulus_only", live_every=5)
results["stimulus_only"] = (design_s, history_s, state_s)

for config_name in ["materials_only", "materials_stimulus"]:
  design, history, state = run_config(
    config_name, init_design_vars=design_s, live_every=5
  )
  results[config_name] = (design, history, state)

for config_name, (design, history, state) in results.items():
  print(f"{config_name:22s} mean u_y = {history[-1] * 1e3:.4f} mm")

### Results

In [ ]:
fig, (ax_shape, ax_conv) = plt.subplots(1, 2, figsize=(12, 4.5))
for config_name, (design, history, state) in results.items():
  coords = deformed_coords(state[0])
  ax_shape.plot(coords[:, 0] * 1e3, coords[:, 1] * 1e3, lw=2, label=config_name)
  ax_conv.plot(np.array(history) * 1e3, label=config_name)

ax_shape.add_patch(
  plt.Rectangle(
    (
      (obstacle_centre[0] - obstacle_span_x / 2) * 1e3,
      (obstacle_centre[1] - obstacle_span_y / 2) * 1e3,
    ),
    obstacle_span_x * 1e3,
    obstacle_span_y * 1e3,
    color="0.7",
  )
)
ax_shape.set_aspect("equal")
ax_shape.set_xlabel("$x$ (mm)")
ax_shape.set_ylabel("$y$ (mm)")
ax_shape.legend()
ax_conv.set_xlabel("MMA epoch")
ax_conv.set_ylabel(r"mean $u_y$ (mm)")
ax_conv.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Magnetization angle along the rod. Sign reversals near the clamp are the
# signature of co-design: the rod bends away first to gain curvature.
fig, ax = plt.subplots(figsize=(6, 4))
for config_name, (design, _, state) in results.items():
  remnant = state[1]
  angle = np.degrees(
    np.arctan2(np.asarray(remnant)[:, 0, 1], np.asarray(remnant)[:, 0, 0])
  )
  ax.plot(np.asarray(rod.elem_centers)[:, 0] * 1e3, angle, lw=1.5, label=config_name)
ax.set_xlabel("$x$ (mm)")
ax.set_ylabel(r"in-plane $B_r$ angle (deg)")
ax.legend()
plt.show()

### Peak contact overlap

In [ ]:
# Positive means the obstacle is genuinely engaged. Zero means it sits outside the
# rod's path and all three configurations reduce to the unobstructed problem, in
# which a uniform axial B_r is already optimal and co-design buys nothing.
for config_name, (design, _, state) in results.items():
  coords = deformed_coords(state[0])
  centres = jnp.asarray(0.5 * (coords[:-1] + coords[1:]))
  obj_trans, obj_rot_mat = obstacle.pose_at_step(0)
  phi = np.asarray(obstacle.compute_sdf(centres, obj_trans, obj_rot_mat))
  print(f"{config_name:22s} peak overlap = {np.max(contact_radii - phi) * 1e3:+.4f} mm")